In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install Flask pandas pyngrok

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import folium
from IPython.display import display, HTML
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
import warnings
import os
import json
from pyngrok import conf, ngrok
import getpass
from flask import Flask, render_template, jsonify
import sys

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DataSci 209/icmcis-drone-tracking/icmcis-drone-detection (2)/train/train/Scenario_1_1/ALVIRA_scenario.csv")
df.head(10)

,total in seconds,datetime(utc),channels,AlviraPotentialDronePlots_timestamp,AlviraPotentialDronPlot_id,AlviraPotentialDronPlot_rcs,AlviraPotentiaDronPlotsPlot_timestamp,AlviraPotentiaDronPlotPlotPosition_altitude,AlviraPotentialDronPlotsPlotPosition_latitude,AlviraPotentialDronPlotsPlotPosition_longitude,...,AlviraSystemStatusSensorPosition_Latitude,AlviraSystemStatusSensorPosition_Longitude,AlviraSystemStatusSensorPosition_Altitude,AlviraSystemStatusSensorStatusOrientetation_Azimuth,AlviraSystemStatusSensorStatusOrientetation_Elevation,AlviraSystemStatusSensorStatus_SensorType,AlviraSystemStatusSensorStatusBlankSector_Angle,AlviraSystemStatusSensorStatusBlankSector_Span,AlviraSystemStatusSensorStatusProcessing_Sensitivity,SystemStatusSensorStatus_Messages
0,1601381457,2020-09-29 12:10:57,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1601381459,2020-09-29 12:10:59,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1601381460,2020-09-29 12:11:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1601381461,2020-09-29 12:11:01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1601381463,2020-09-29 12:11:03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1601381464,2020-09-29 12:11:04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1601381465,2020-09-29 12:11:05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1601381467,2020-09-29 12:11:07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1601381468,2020-09-29 12:11:08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1601381469,2020-09-29 12:11:09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
!mkdir -p templates
print("Created 'templates' directory.")

Created 'templates' directory.


In [ ]:
authtoken = getpass.getpass("Enter your ngrok authtoken: ")
conf.get_default().auth_token = authtoken

Enter your ngrok authtoken: ··········


In [ ]:
%%writefile app.py
from flask import Flask, render_template, jsonify
import pandas as pd
import json
import os
import sys

app = Flask(__name__)

# change csv path if needed
csv_path = "/content/drive/MyDrive/Colab Notebooks/DataSci 209/icmcis-drone-tracking/icmcis-drone-detection (2)/train/train/Scenario_1_1/ALVIRA_scenario.csv"
df = pd.read_csv(csv_path)

df["datetime(utc)"] = pd.to_datetime(df["datetime(utc)"], errors="coerce")
df = df.dropna(subset=["datetime(utc)"])
df = df.sort_values("datetime(utc)")
df = df.dropna(subset=[
    "AlviraTracksTrackPosition_Latitude",
    "AlviraTracksTrackPosition_Longitude",
    "AlviraTracksTrackPosition_Altitude"
])

# time conversion to flask standard
df['datetime(utc)'] = df['datetime(utc)'].dt.strftime('%Y-%m-%d %H:%M:%S.%f')

# float type conversion
df['AlviraTracksTrackPosition_Altitude'] = df['AlviraTracksTrackPosition_Altitude'].astype(float)
df['AlviraTracksTrackPosition_Latitude'] = df['AlviraTracksTrackPosition_Latitude'].astype(float)
df['AlviraTracksTrackPosition_Longitude'] = df['AlviraTracksTrackPosition_Longitude'].astype(float)


min_time = df["datetime(utc)"].min()
max_time = df["datetime(utc)"].max()
min_altitude = float(df["AlviraTracksTrackPosition_Altitude"].min())
max_altitude = float(df["AlviraTracksTrackPosition_Altitude"].max())

# put a buffer into long & lat to have it easier on the eyes
buffer = 0.001

min_latitude = float(df["AlviraTracksTrackPosition_Latitude"].min())
max_latitude = float(df["AlviraTracksTrackPosition_Latitude"].max())
min_longitude = float(df["AlviraTracksTrackPosition_Longitude"].min())
max_longitude = float(df["AlviraTracksTrackPosition_Longitude"].max())

# scale to the location of the drone
min_latitude_buffered = min_latitude - buffer
max_latitude_buffered = max_latitude + buffer
min_longitude_buffered = min_longitude - buffer
max_longitude_buffered = max_longitude + buffer



@app.route('/')
def index():
    initial_data = df.to_dict(orient='records')
    return render_template(
        'index.html',
        drone_data_json=json.dumps(initial_data),
        min_time=min_time,
        max_time=max_time,
        min_altitude=min_altitude,
        max_altitude=max_altitude,
        min_latitude=min_latitude_buffered,
        max_latitude=max_latitude_buffered,
        min_longitude=min_longitude_buffered,
        max_longitude=max_longitude_buffered,
        total_frames=len(df)
    )

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=False)

Overwriting app.py


In [ ]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Drone Altitude & Flight Path (Flask + Plotly.js Map)</title>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        body {
            font-family: verdana;
            margin: 20px;
            background-color: #f0f2f5;
            color: #333;
        }
        h1 {
            color: #2c3e50;
            text-align: center;
            margin-bottom: 30px;
        }
        .controls {
            margin-bottom: 20px;
            background-color: #ffffff;
            padding: 15px;
            border-radius: 8px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            display: flex;
            align-items: center;
            gap: 15px;
            justify-content: center;
        }
        label {
            font-weight: bold;
            color: #555;
            white-space: nowrap;
        }
        input[type="range"] {
            flex-grow: 1;
            margin: 0;
            height: 25px;
            -webkit-appearance: none;
            width: 100%;
            background: #d3d3d3;
            outline: none;
            opacity: 0.7;
            -webkit-transition: .2s;
            transition: opacity .2s;
            border-radius: 5px;
        }
        input[type="range"]:hover {
            opacity: 1;
        }
        input[type="range"]::-webkit-slider-thumb {
            -webkit-appearance: none;
            appearance: none;
            width: 25px;
            height: 25px;
            background: #007bff;
            cursor: pointer;
            border-radius: 50%;
            box-shadow: 0 1px 3px rgba(0,0,0,0.2);
        }
        input[type="range"]::-moz-range-thumb {
            width: 25px;
            height: 25px;
            background: #007bff;
            cursor: pointer;
            border-radius: 50%;
            box-shadow: 0 1px 3px rgba(0,0,0,0.2);
        }
        #currentSliderTime {
            font-weight: bold;
            min-width: 200px;
            text-align: left;
            color: #007bff;
            padding-left: 10px;
        }

        .charts-container {
            display: flex;
            gap: 20px;
            flex-wrap: wrap;
            justify-content: center;
        }

        #altitudeChart, #flightPathMap {
            flex: 1;
            min-width: 450px;
            height: 600px;
            background-color: #ffffff;
            border-radius: 8px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            padding: 10px;
        }

        @media (max-width: 960px) {
            .charts-container {
                flex-direction: column;
                align-items: center;
            }
            #altitudeChart, #flightPathMap {
                min-width: 90%;
                max-width: 600px;
            }
        }
    </style>
</head>
<body>
    <h1>ARCUS Drone Tracker Dashboard</h1>

    <div class="controls">
        <label for="timeSlider">Time Point:</label>
        <input type="range" id="timeSlider" min="0" max="{{ total_frames - 1 }}" value="0">
        <span id="currentSliderTime"></span>
    </div>

    <div class="charts-container">
        <div id="altitudeChart"></div>
        <div id="flightPathMap"></div>
    </div>

<script>
    // data from flask
    const droneData = {{ drone_data_json | safe }};
    const minTime = "{{ min_time }}";
    const maxTime = "{{ max_time }}";
    const minAltitude = {{ min_altitude }};
    const maxAltitude = {{ max_altitude }};
    const totalFrames = {{ total_frames }};

    const minLatitude = {{ min_latitude }};
    const maxLatitude = {{ max_latitude }};
    const minLongitude = {{ min_longitude }};
    const maxLongitude = {{ max_longitude }};
    const timeSlider = document.getElementById('timeSlider');
    const currentSliderTimeDisplay = document.getElementById('currentSliderTime');

    // link data to plotly
    const allTimes = droneData.map(d => d['datetime(utc)']);
    const allAltitudes = droneData.map(d => d.AlviraTracksTrackPosition_Altitude);
    const allLongitudes = droneData.map(d => d.AlviraTracksTrackPosition_Longitude);
    const allLatitudes = droneData.map(d => d.AlviraTracksTrackPosition_Latitude);

    // altitude map visualization
    const initialTraceLine2D = {
        x: [], y: [],
        mode: 'lines',
        line: { color: 'steelblue', width: 2 },
        name: 'Altitude Path'
    };
    const initialTraceMarker2D = {
        x: [], y: [],
        mode: 'markers',
        marker: { size: 10, color: '#FF6B6B', symbol: 'circle' },
        name: 'Current Position'
    };
    const initialLayout2D = {
        title: `Altitude Over Time: ${allTimes[0] || 'Loading...'}`,
        xaxis: {
            title: 'Time',
            range: [minTime, maxTime],
            type: 'date'
        },
        yaxis: {
            title: 'Altitude',
            range: [minAltitude * 0.95, maxAltitude * 1.05]
        },
        showlegend: true,
        legend: { x: 0.01, y: 0.99 },
        margin: { l: 50, r: 20, b: 50, t: 70 }
    };
    Plotly.newPlot('altitudeChart', [initialTraceLine2D, initialTraceMarker2D], initialLayout2D);

    // flight path visualization
    const avgLatitude = (minLatitude + maxLatitude) / 2;
    const avgLongitude = (minLongitude + maxLongitude) / 2;

    const latDiff = maxLatitude - minLatitude;
    const lonDiff = maxLongitude - minLongitude;
    const maxGeographicDiff = Math.max(latDiff, lonDiff);

    let initialMapZoom = 16;
    if (maxGeographicDiff > 0.01) initialMapZoom = 14;
    if (maxGeographicDiff > 0.1) initialMapZoom = 12;
    if (maxGeographicDiff > 0.5) initialMapZoom = 10;
    if (maxGeographicDiff > 1) initialMapZoom = 8;
    if (maxGeographicDiff === 0) initialMapZoom = 17;

    const initialTraceLineMap = {
        type: 'scattermapbox',
        lon: [], lat: [],
        mode: 'lines',
        line: { color: 'steelblue', width: 3 },
        name: 'Flight Path',
        text: [],
        hoverinfo: 'text+name'
    };
    const initialTraceMarkerMap = {
        type: 'scattermapbox',
        lon: [], lat: [],
        mode: 'markers',
        marker: { size: 12, color: '#FF6B6B', symbol: 'circle' },
        name: 'Current Position',
        text: [],
        hoverinfo: 'text+name'
    };

    const initialLayoutMap = {
        title: `Drone Flight Path on Map: ${allTimes[0] || 'Loading...'}`,
        mapbox: {
            style: 'carto-positron',
            center: { lat: avgLatitude, lon: avgLongitude },
            zoom: initialMapZoom
        },
        margin: { l: 0, r: 0, b: 0, t: 70 },
        showlegend: true,
        legend: { x: 0.01, y: 0.99 }
    };
    Plotly.newPlot('flightPathMap', [initialTraceLineMap, initialTraceMarkerMap], initialLayoutMap);

    // simultaneously updating both visualizations at the same time
    function updatePlots(index) {
        if (droneData.length === 0) {
            console.error("updatePlots called but droneData is empty.");
            return;
        }
        if (index >= droneData.length || index < 0) {
            console.warn("Attempted to update plot with out-of-bounds index:", index);
            return;
        }

        const currentPoint = droneData[index];

        // time format for cleaner display
        const timeOnly = new Date(currentPoint['datetime(utc)']).toLocaleTimeString('en-GB', {
            hour: '2-digit',
            minute: '2-digit',
            second: '2-digit',
            hour12: false
        });

        // --- Update 2D Altitude Plot ---
        const updatedLineX2D = allTimes.slice(0, index + 1);
        const updatedLineY2D = allAltitudes.slice(0, index + 1);
        const updatedMarkerX2D = [currentPoint['datetime(utc)']];
        const updatedMarkerY2D = [currentPoint.AlviraTracksTrackPosition_Altitude];

        Plotly.restyle('altitudeChart', {
            x: [updatedLineX2D, updatedMarkerX2D],
            y: [updatedLineY2D, updatedMarkerY2D]
        }, [0, 1]);

        Plotly.relayout('altitudeChart', {
            title: `Altitude Over Time at: ${timeOnly}`
        });

        const updatedLineLonMap = allLongitudes.slice(0, index + 1);
        const updatedLineLatMap = allLatitudes.slice(0, index + 1);
        const updatedLineTextMap = allAltitudes.slice(0, index + 1).map(alt => `Altitude: ${alt.toFixed(2)}m`);

        const updatedMarkerLonMap = [currentPoint.AlviraTracksTrackPosition_Longitude];
        const updatedMarkerLatMap = [currentPoint.AlviraTracksTrackPosition_Latitude];
        const updatedMarkerTextMap = [`Altitude: ${currentPoint.AlviraTracksTrackPosition_Altitude.toFixed(2)}m`];

        Plotly.restyle('flightPathMap', {
            lon: [updatedLineLonMap, updatedMarkerLonMap],
            lat: [updatedLineLatMap, updatedMarkerLatMap],
            text: [updatedLineTextMap, updatedMarkerTextMap]
        }, [0, 1]);

        Plotly.relayout('flightPathMap', {
            title: `Drone Flight Path: ${timeOnly}`
        });

        currentSliderTimeDisplay.textContent = `Time: ${timeOnly}`;
        timeSlider.value = index;
    }

    timeSlider.addEventListener('input', (event) => {
        const selectedIndex = parseInt(event.target.value);
        updatePlots(selectedIndex);
    });

    if (droneData.length > 0) {
        updatePlots(0);
    } else {
        document.getElementById('altitudeChart').innerHTML = "<p style='text-align: center; color: red; font-weight: bold;'>Error: No drone data loaded for 2D plot.</p>";
        document.getElementById('flightPathMap').innerHTML = "<p style='text-align: center; color: red; font-weight: bold;'>Error: No drone data loaded for map plot.</p>";
    }
</script>

</body>
</html>

Overwriting templates/index.html


In [ ]:
import threading
import time
import sys
from pyngrok import ngrok

!fuser -k 5000/tcp || true
ngrok.kill()

def run_flask_app():
    if '/content/' not in sys.path:
        sys.path.insert(0, '/content/')
    from app import app
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

thread = threading.Thread(target=run_flask_app)
thread.daemon = True
thread.start()
time.sleep(5)

# ngrok setup
try:
    public_url = ngrok.connect(5000)
    print(f"Flask App URL: {public_url}")
except Exception as e:
    print(f"Error connecting ngrok: {e}")
    sys.exit(1)

try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("Shutting down ngrok tunnel...")
    ngrok.kill()
    print("Tunnel shut down.")

 * Serving Flask app 'app'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


Flask App URL: NgrokTunnel: "https://c703-34-80-147-122.ngrok-free.app" -> "http://localhost:5000"


INFO:werkzeug:127.0.0.1 - - [08/Jun/2025 01:29:39] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [08/Jun/2025 01:29:41] "GET /favicon.ico HTTP/1.1" 404 -
